In [ ]:

annotations_path = "/home/work/yuna/VLMEval/data/v2_mscoco_val2014_annotations.json"
with open(os.path.join(annotations_path), 'r') as f: 
    annotations = json.load(f)['annotations'] 

with open(os.path.join("/home/work/yuna/VLMEval/data/v2_OpenEnded_mscoco_val2014_questions.json"), 'r') as f: 
    annotations = json.load(f)

In [ ]:
from glob import glob
import pandas as pd
import os

# Ensure output directory exists
output_dir = '/home/work/yuna/hpa/results/closed_source-processed'
os.makedirs(output_dir, exist_ok=True)  # Create directory if it doesn't exist

dfs = []
for f in glob('/home/work/yuna/hpa/results/closed_source/*.jsonl'):
    # Read JSONL file into DataFrame
    df = pd.read_json(f, lines=True)
    filename = f.split('/')[-1][:-6]  # Remove '.jsonl'
    output_file = os.path.join(output_dir, f"{filename}.jsonl")

    # Process DataFrame
    df = get_question_id(df)  # Ensure this function returns a DataFrame

    if 'answer' in df.columns:
        df.rename(columns={'answer': 'raw_output'}, inplace=True)

    if 'model' not in df.columns:
        df['model'] = filename
    # print(df)

    # Open output file
    with open(output_file, "w", encoding="utf-8") as fout:
        for i, row in df.iterrows():
            qid = int(row.get("question_id", 0))  # Default to 0 if missing
            ans = get_answer(qid)  # Ensure this function is defined
            processed_output = postprocessor.postprocess_answer(row.get("raw_output", ""))
            acc = vqa_score(processed_output, ans)  # Ensure this function is defined
            score= 0 
            try:
                score = float(answer_similarity(processed_output, ans))  # Ensure this function is defined
            except Exception as e:
                print(f"Error processing row {row}")
                continue
            item = {
                "qid": qid,
                "question": row.get("question", "").strip(),
                "question_type": question_type.get(qid, ""),  # Ensure question_type is defined
                "model": row.get("model", "").strip(),
                "raw_output": row.get("raw_output", "").strip(),
                "processed_output": processed_output,
                "acc": acc,
                "score": float(score)
            }
            fout.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"Saved to: {output_file}")
    dfs.append(df)